In [10]:
import csv
from jobspy import scrape_jobs

# Define the specific columns you want
desired_columns = ["job_url","title","company","location","date_posted","job_type","is_remote","job_level","emails","description","company_logo"]

jobs = scrape_jobs(
    site_name=["linkedin"], # "glassdoor", "bayt", "naukri", "bdjobs"
    search_term="computer vision",
    google_search_term="computer vision jobs near Europe since yesterday",
    location="switzerland",
    results_wanted=100,
    hours_old=24,
    
    linkedin_fetch_description=True # gets more info such as description, direct job url (slower)
    # proxies=["208.195.175.46:65095", "208.195.175.45:65095", "localhost"],
)

print(f"Found {len(jobs)} jobs")

# Filter to only the columns you want immediately after scraping
available_desired_cols = [col for col in desired_columns if col in jobs.columns]
missing_cols = [col for col in desired_columns if col not in jobs.columns]

if missing_cols:
    print(f"⚠️ Missing columns: {missing_cols}")

# Keep only the desired columns
jobs_filtered = jobs[available_desired_cols].copy()

print(f"📊 Filtered to {len(available_desired_cols)} columns: {available_desired_cols}")
print(jobs_filtered.head())

# Save the filtered data
jobs_filtered.to_csv("jobs_filtered.csv", quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)

Found 17 jobs
              id      site                                        job_url  \
0  li-4293921160  linkedin  https://www.linkedin.com/jobs/view/4293921160   
1  li-4296033986  linkedin  https://www.linkedin.com/jobs/view/4296033986   
2  li-4162279909  linkedin  https://www.linkedin.com/jobs/view/4162279909   
3  li-4295686547  linkedin  https://www.linkedin.com/jobs/view/4295686547   
4  li-4295317410  linkedin  https://www.linkedin.com/jobs/view/4295317410   

                                      job_url_direct  \
0  https://join.com/companies/pixelplus/14785111-...   
1  https://www.twine.net/projects/b8y1o0-aneira-d...   
2  https://job-boards.greenhouse.io/isomorphiclab...   
3  https://ethonai.recruitee.com/o/causal-ai-engi...   
4                                                NaN   

                                             title          company  \
0                                      AI Engineer    Pixel Plus AG   
1                AI Developer – Freelance (R

In [ ]:
# Convert to LibreOffice-compatible formats
import pandas as pd

print(f"Converting {len(jobs)} jobs to LibreOffice formats...")

# Clean the data for LibreOffice compatibility
jobs_clean = jobs.copy()

# Clean text fields to remove problematic characters
text_columns = ['title', 'company', 'location', 'description', 'company_description']
for col in text_columns:
    if col in jobs_clean.columns:
        jobs_clean[col] = jobs_clean[col].astype(str).str.replace('\n', ' ').str.replace('\r', ' ').str.replace('"', '""')

# 1. LibreOffice CSV (UTF-8 with BOM, semicolon separated)
jobs_clean.to_csv(
    "jobs_libreoffice.csv", 
    encoding='utf-8-sig',
    sep=';',
    quoting=csv.QUOTE_ALL,
    index=False,
    lineterminator='\n'
)

# 2. Excel format (works perfectly in LibreOffice)
jobs_clean.to_excel("jobs_libreoffice.xlsx", index=False, engine='openpyxl')

# 3. Create summary view with key columns
summary_columns = ['title', 'company', 'location', 'job_type', 'is_remote', 'date_posted', 'job_url', 'description']
available_cols = [col for col in summary_columns if col in jobs.columns]
jobs_summary = jobs[available_cols].copy()

# Add short description preview
if 'description' in jobs_summary.columns:
    jobs_summary['description'] = jobs_summary['description'].astype(str).str[:300] + '...'

jobs_summary.to_csv(
    "jobs_summary.csv",
    encoding='utf-8-sig',
    sep=';',
    quoting=csv.QUOTE_ALL,
    index=False
)

In [ ]:
import time

search_terms = ["AI Engineer", "AI Developer", "Machine Learning", "Data Scientist", "Computer Vision", "Deep Learning", "NLP", "AI Architect"]
target_locations = ["Sweden", "Switzerland", "Norway", "Ireland", "Germany", "Belgium", "Netherlands", "Luxembourg", "Denmark"]
desired_columns = ["job_url","title","company","location","date_posted","job_type","is_remote","job_level","emails","description","company_logo"]

def run_search(search_terms, target_locations):

    all_results = []

    for search_term in search_terms:
        for location in target_locations:
            
            try:
                jobs = scrape_jobs(
                    site_name=["linkedin", "glassdoor", "indeed"],  # Multiple sites
                    search_term=search_term,
                    google_search_term=f"{search_term} jobs {location} Europe recent",
                    location=location,
                    results_wanted=100,  # Smaller batches for reliability
                    hours_old=24,  # 3 days instead of 24 hours
                    
                    linkedin_fetch_description=True,  # Keep detailed descriptions
                    # proxies=["208.195.175.46:65095"] if needed  # Uncomment if you have proxies
                )
                
                if len(jobs) > 0:
                    jobs['search_query'] = search_term
                    jobs['target_location'] = location
                    all_results.append(jobs)
                    print(f"Found {len(jobs)} jobs")
                else:
                    print(f"No results for this combination")
                    
            except Exception as e:
                print(f"Error: {str(e)}")
                continue
            
            # Rate limiting - important for reliability!
            time.sleep(3)
    
    # Combine and clean results
    if all_results:
        final_jobs = pd.concat(all_results, ignore_index=True)
        
        # Remove duplicates by job URL
        final_jobs = final_jobs.drop_duplicates(subset=['job_url'], keep='first')

        return final_jobs
    else:
        print("No results found")
        return pd.DataFrame()

def extract_job_variables(jobs_df, columns_wanted):
    """
    Extract only the specified columns from the jobs dataframe
    """
    available_columns = [col for col in columns_wanted if col in jobs_df.columns]
    missing_columns = [col for col in columns_wanted if col not in jobs_df.columns]
    
    if missing_columns:
        print(f"Missing columns: {missing_columns}")
    
    # Extract only the available desired columns
    filtered_jobs = jobs_df[available_columns].copy()
    
    return filtered_jobs


jobs = run_search()
filtered_jobs = extract_job_variables(jobs, desired_columns)

🔍 Searching: 'computer vision engineer' in switzerland
  ✅ Found 33 jobs
🔍 Searching: 'computer vision engineer' in germany
  ✅ Found 58 jobs
🔍 Searching: 'machine learning computer vision' in switzerland
  ✅ Found 22 jobs
🔍 Searching: 'machine learning computer vision' in germany
  ✅ Found 42 jobs

🎯 FINAL RESULTS: 136 unique jobs found
📊 Breakdown by location: {'germany': 85, 'switzerland': 51}
✅ Improved search function ready!
💡 Uncomment the last line to run it with optimized parameters


/tmp/ipykernel_306949/1676593207.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_jobs = pd.concat(all_results, ignore_index=True)


In [ ]:
filtered_jobs

,id,site,job_url,job_url_direct,title,company,location,date_posted,job_type,salary_source,...,company_revenue,company_description,skills,experience_range,company_rating,company_reviews_count,vacancy_count,work_from_home_type,search_term_used,search_location
0,gd-1009865792914,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,"Industrial PhD student, AI in clinical workflo...",Sectra AB,Linköping,2025-09-05,NaN,None,...,None,None,None,None,None,None,None,None,computer vision,Sweden
1,gd-1009866385184,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,Rendering Software Engineer - C++,Electronic Arts (EA),Stockholm,2025-09-05,NaN,None,...,None,None,None,None,None,None,None,None,computer vision,Sweden
2,gd-1009866477958,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,Staff/Senior Machine Learning Engineer,Voi Technology,Stockholm,2025-09-05,NaN,None,...,None,None,None,None,None,None,None,None,computer vision,Sweden
3,gd-1009866478096,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,Group IT Infrastructure Lead,Allgon,Västra Frölunda,2025-09-05,NaN,None,...,None,None,None,None,None,None,None,None,computer vision,Sweden
4,gd-1009866547542,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,Research assistant with technical background,Karolinska Institutet (KI),Huddinge,2025-09-05,NaN,None,...,None,None,None,None,None,None,None,None,computer vision,Sweden
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,li-4285637317,linkedin,https://www.linkedin.com/jobs/view/4285637317,https://careersatagoda.com/job/5950274-senior-...,"Senior Data Scientist (Bangkok based, relocati...",Agoda,"Oslo, Oslo, Norway",NaN,fulltime,None,...,None,None,None,None,None,None,None,None,computer vision,Norway
95,li-4285635521,linkedin,https://www.linkedin.com/jobs/view/4285635521,https://careersatagoda.com/job/5432621-lead-st...,"Lead/ Staff Data Scientist (Bangkok based, rel...",Agoda,"Oslo, Oslo, Norway",NaN,fulltime,None,...,None,None,None,None,None,None,None,None,computer vision,Norway
96,li-4285641202,linkedin,https://www.linkedin.com/jobs/view/4285641202,https://careersatagoda.com/job/5432627-lead-ds...,"Lead DS (Data Scientist) Bangkok based, Reloca...",Agoda,"Oslo, Oslo, Norway",NaN,fulltime,None,...,None,None,None,None,None,None,None,None,computer vision,Norway
97,li-4285634848,linkedin,https://www.linkedin.com/jobs/view/4285634848,https://careersatagoda.com/job/6030012-lead-da...,"Lead Data Science (Bangkok based, Relocation p...",Agoda,"Oslo, Oslo, Norway",NaN,fulltime,None,...,None,None,None,None,None,None,None,None,computer vision,Norway
